In [61]:
import pandas as pd
import numpy as np
from pathlib import Path

In [62]:
PATH = Path.cwd()
PARENT = PATH.parent

dataset_path = (PARENT / "data" / "scored_articles.csv")
df = pd.read_csv(dataset_path, index_col= False)
print(df.head().to_string())

   id                      article_hash                                                                                         title                                                                                                                                                                                                                                                                                                                                                                                                                                     summary   source_name    category                                                                                                                                                                                                                                                                                                                                                                                                                          url

In [63]:
# Helper functions for token overlap and Jaccard similarity
def calculate_jaccard_similarity(row):
    t_words = set(str(row["title"]).lower().split())
    s_words = set(str(row["summary"]).lower().split())

    union = t_words.union(s_words)
    if not union:
        return 0.0
    return len(t_words.intersection(s_words)) / len(union)


def calculate_overlap_ratio(row):
    """Calculates how much of the title's vocabulary appears in the summary."""
    t_words = set(str(row["title"]).lower().split())
    s_words = set(str(row["summary"]).lower().split())

    if not t_words:
        return 0.0
    return len(t_words.intersection(s_words)) / len(t_words)

In [ ]:
#Creating new features

def new(df):
    #Article age
    df["published_at"] = pd.to_datetime(df["published_at"], utc= True, errors='coerce', format= 'mixed')
    df["collected_at"] = pd.to_datetime(df["collected_at"], utc= True, errors='coerce')

    df["article_age"] = np.round((df["collected_at"] - df["published_at"]).dt.total_seconds() / 3600, 1)

    #Extract the day
    df["publish_day"] = df["published_at"].dt.strftime("%a")

    #Extract the time
    df["publish_hour"] = df["published_at"].dt.hour

    #Is weekend
    df["is_weekend"] = df["publish_day"].isin(['Sun', 'Sat']).astype(int)

    #text complexity features
    df["title_len"] = df["title"].str.len()
    df["summary_len"] = df["summary"].str.len()

    df["title_word_count"] = df["title"].str.split().str.len()
    df["summary_word_count"] = df["summary"].str.split().str.len()

    df["title_to_summary_ratio"] = df["title_len"] / (df["summary_len"] + 1)

    df["caps_ratio"] = df["title"].apply(
        lambda t: sum(1 for c in str(t) if c.isupper()) / max(len(str(t)), 1)
    )

    df["punctuation_triggers"] = df["title"].str.count(r"[!?]")

    df["has_digits"] = df['title'].str.contains(r'\d').astype(int)

    df["source_category_interaction"] = df["source_name"] + "_" + df["category"]

    df["text_similarity"] = df.apply(calculate_jaccard_similarity, axis=1)
    df["overlap_ratio"] = df.apply(calculate_overlap_ratio, axis=1)

    return df



Total missing: 508

Unique sources causing missing dates:
source_name
Phys.org - latest science and technology news stories    219
www.espn.com - TOP                                       180
All Top News -- ScienceDaily                              72
Politics                                                  37
Name: count, dtype: int64

Sample raw published_at values:
Series([], Name: published_at, dtype: datetime64[us, UTC])

How many are completely empty/None in published_at?
508
